# 🚀 FAST Android Emulator - MAX GPU

1. Enable GPU: Runtime → Change runtime → T4 GPU
2. Run all cells
3. Choose phone → Get link → Done!

In [ ]:
# Step 1: Quick Install (90 seconds)
%%bash
export DEBIAN_FRONTEND=noninteractive
apt-get update -qq && apt-get install -y -qq wget unzip openjdk-11-jdk xvfb x11vnc > /dev/null 2>&1
cd /content
git clone -q --depth=1 https://github.com/novnc/noVNC.git
git clone -q --depth=1 https://github.com/novnc/websockify noVNC/utils/websockify
echo "✅ Done!"

In [ ]:
# Step 2: Android SDK (2 min)
%%bash
cd /content
wget -q https://dl.google.com/android/repository/commandlinetools-linux-9477386_latest.zip
mkdir -p android-sdk/cmdline-tools
unzip -q commandlinetools-linux-9477386_latest.zip -d android-sdk/cmdline-tools
mv android-sdk/cmdline-tools/cmdline-tools android-sdk/cmdline-tools/latest
rm *.zip
echo "✅ SDK ready!"

In [ ]:
# Step 3: Install Emulator (3 min)
import os
os.environ['ANDROID_HOME'] = '/content/android-sdk'
os.environ['PATH'] = f"{os.environ['PATH']}:/content/android-sdk/cmdline-tools/latest/bin:/content/android-sdk/platform-tools:/content/android-sdk/emulator"

!yes | /content/android-sdk/cmdline-tools/latest/bin/sdkmanager --licenses > /dev/null 2>&1
!/content/android-sdk/cmdline-tools/latest/bin/sdkmanager "platform-tools" "platforms;android-33" "emulator" "system-images;android-33;google_apis;x86_64"
print("✅ Emulator installed!")

In [ ]:
# Step 4: Choose Phone
PHONES = {
    "1": {"name": "Pixel 7 Pro", "w": 1440, "h": 3120, "dpi": 560, "dev": "pixel_7_pro"},
    "2": {"name": "Galaxy S23", "w": 1080, "h": 2340, "dpi": 425, "dev": "pixel_6"},
    "3": {"name": "Pixel 6", "w": 1080, "h": 2400, "dpi": 411, "dev": "pixel_6"},
    "4": {"name": "OnePlus", "w": 1080, "h": 2400, "dpi": 440, "dev": "pixel_5"},
}

print("📱 PHONES:\n" + "="*40)
for k, v in PHONES.items():
    print(f"{k}. {v['name']} ({v['w']}x{v['h']})")
    
choice = input("\nPick (1-4): ").strip()
P = PHONES.get(choice, PHONES["3"])
print(f"\n✅ {P['name']} selected!")
%store P

In [ ]:
# Step 5: Create Device (15 sec)
%store -r P
import os
os.environ['ANDROID_HOME'] = '/content/android-sdk'

!/content/android-sdk/cmdline-tools/latest/bin/avdmanager create avd --force --name F --package "system-images;android-33;google_apis;x86_64" --device "{P['dev']}"

cfg = f"""hw.lcd.width={P['w']}\nhw.lcd.height={P['h']}\nhw.lcd.density={P['dpi']}\nhw.gpu.enabled=yes\nhw.gpu.mode=host\nhw.ramSize=4096\n"""
with open('/root/.android/avd/F.avd/config.ini', 'a') as f:
    f.write(cfg)
print(f"✅ {P['name']} ready!")

In [ ]:
# Step 6: Start Display
import subprocess, time, os
os.environ['DISPLAY'] = ':99'

subprocess.Popen(['Xvfb', ':99', '-screen', '0', '1920x1080x24', '-ac', '+extension', 'GLX', '+render'], stdout=subprocess.PIPE, stderr=subprocess.PIPE)
time.sleep(3)

subprocess.Popen(['x11vnc', '-display', ':99', '-nopw', '-forever', '-xkb', '-shared', '-repeat', '-bg'], stdout=subprocess.PIPE, stderr=subprocess.PIPE)
time.sleep(3)

subprocess.Popen(['/content/noVNC/utils/novnc_proxy', '--vnc', 'localhost:5900', '--listen', '6080'], stdout=subprocess.PIPE, stderr=subprocess.PIPE)
time.sleep(3)
print("✅ Display ready!")

In [ ]:
# Step 7: Get Link (Cloudflare - NO SIGNUP NEEDED!)
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x cloudflared-linux-amd64

import subprocess, time, re

print("🔗 Creating public link...\n")
proc = subprocess.Popen(['./cloudflared-linux-amd64', 'tunnel', '--url', 'http://localhost:6080'], 
                        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

for _ in range(30):
    line = proc.stdout.readline()
    if 'trycloudflare.com' in line:
        url = re.search(r'https://[\w-]+\.trycloudflare\.com', line)
        if url:
            link = url.group(0)
            print("\n" + "="*60)
            print("🎉 YOUR LINK:")
            print("="*60)
            print(f"\n🌐 {link}/vnc.html\n")
            print("="*60)
            print("⚡ Phone boots in 60 sec - click link now!\n")
            break
    time.sleep(1)

In [ ]:
# Step 8: BOOT (MAX GPU)
import subprocess, os, time
os.environ['ANDROID_HOME'] = '/content/android-sdk'
os.environ['DISPLAY'] = ':99'
os.environ['PATH'] = f"{os.environ['PATH']}:/content/android-sdk/emulator:/content/android-sdk/platform-tools"
%store -r P

print(f"🚀 Booting {P['name']} with MAX GPU...\n")

subprocess.Popen([
    '/content/android-sdk/emulator/emulator', '-avd', 'F',
    '-gpu', 'host', '-accel', 'on', '-no-audio', '-no-boot-anim',
    '-writable-system', '-no-snapshot-load',
    '-qemu', '-smp', '4', '-m', '4096'
], stdout=subprocess.PIPE, stderr=subprocess.PIPE)

print("⚡ Starting with MAXIMUM performance...")
for i in range(60):
    r = subprocess.run(['/content/android-sdk/platform-tools/adb', 'devices'], capture_output=True, text=True)
    if 'emulator-5554' in r.stdout:
        print("\n✅ Device online!")
        break
    if i % 10 == 0:
        print(f"⏳ {i}s...")
    time.sleep(1)

!/content/android-sdk/platform-tools/adb wait-for-device
!/content/android-sdk/platform-tools/adb shell 'while [ "$(getprop sys.boot_completed)" != "1" ]; do sleep 1; done'

print("\n🎉 READY! Check browser!")

In [ ]:
# Step 9: Upload APK
from google.colab import files
uploaded = files.upload()
apk = list(uploaded.keys())[0]

!/content/android-sdk/platform-tools/adb install -r -t "{apk}"

r = !aapt dump badging "{apk}" | grep package:
if r:
    pkg = r[0].split("name='")[1].split("'")[0]
    !/content/android-sdk/platform-tools/adb shell monkey -p {pkg} -c android.intent.category.LAUNCHER 1
    print(f"\n✅ {pkg} launched!")

In [ ]:
# Screenshot
!/content/android-sdk/platform-tools/adb shell screencap -p /sdcard/s.png
!/content/android-sdk/platform-tools/adb pull /sdcard/s.png .
from google.colab import files
files.download('s.png')

In [ ]:
# Device info
!/content/android-sdk/platform-tools/adb devices
!/content/android-sdk/platform-tools/adb shell wm size
!/content/android-sdk/platform-tools/adb shell wm density